<table><tr>
<td width="76"><div align="center" style="font-size:44px">🕵️</div></td>
<td><h1 style="margin:0">LAB 2 · Encuentra el C2</h1>
<b>Máster en Cyber Threat Intelligence · Módulo 06 · Sesión 44 — Análisis estático de malware</b><br>
⏱️ <b>15 minutos</b> &nbsp;·&nbsp; 🎯 Sacarle a un troyano su servidor de mando y control, sin ejecutarlo</td>
</tr></table>

---

## La situación

En el LAB 1 descubrimos que `curriculum_vitae_2024.pdf` **no es un PDF**: es un ejecutable de
Windows, y VirusTotal dice que es un troyano de control remoto.

Ahora tu jefa te pide lo siguiente:

> *"Vale, es un RAT. Necesito **la dirección del servidor al que se conecta** para bloquearla en
> el firewall y buscar en los logs si alguien más se ha conectado. Y lo necesito ya."*

Un troyano de control remoto tiene que hablar con su **C2** (*Command & Control*): el servidor
desde el que el atacante le da órdenes. Y esa dirección **tiene que estar dentro del fichero**:
el programa necesita saber a dónde llamar.

Nuestro trabajo es encontrarla. Sin ejecutar nada.

### Lo que tienes que entregar
1. 🌐 **El dominio del C2**
2. 🔧 **Dos técnicas del atacante** que hayas deducido de los textos del fichero
3. 🎁 *(Reto)* Una **tabla de IOCs** clasificada por categorías

---
## 🔧 Preparación · ejecuta esta celda SIEMPRE

**Cada laboratorio es un cuaderno distinto y arranca en una máquina nueva.**
Aunque vengas del laboratorio anterior, aquí no hay nada instalado y las muestras
todavía no están. No se comparte nada entre cuadernos.

Pulsa ▶️ en la celda de abajo y espera unos **40 segundos**. Solo hay que hacerlo
una vez por laboratorio.

> 📦 La segunda celda, **PLAN B**, solo hace falta si la primera no consigue las
> muestras. Si la primera termina con el listado de ficheros, ignórala y sigue.


In [ ]:
#@title ▶️ EJECUTA ESTA CELDA (botón ▶ a la izquierda) y espera ~40 segundos { display-mode: "form" }

#@markdown ---
#@markdown **No hace falta que entiendas este código todavía.** Solo prepara el laboratorio:
#@markdown instala las herramientas, descarga las muestras y las descomprime.
#@markdown ---

SAMPLES_URL = "https://github.com/jstnk9/kschool_ejercicios/raw/main/analisis_estatico/muestras_kschool.zip" #@param {type:"string"}
PASSWORD    = "infected" #@param {type:"string"}

import os, glob, subprocess

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("1/4  Instalando herramientas de análisis...")
_sh("pip install -q pyzipper pefile oletools yara-python py-tlsh")
print("     ✔ pyzipper, pefile, oletools, yara-python, tlsh")

print("2/4  Consiguiendo las muestras...")
DEST = "/content/muestras_kschool.zip"

if not SAMPLES_URL.strip():
    # Sin URL configurada: las muestras se suben a mano con la celda de abajo.
    print("     ℹ  Este cuaderno no trae URL de descarga.")
    print("        Ve a la celda de abajo, 'PLAN B', y sube el fichero")
    print("        muestras_kschool.zip que te ha pasado el profesor.")
    print("        Es un paso normal: tarda 10 segundos.")
    ok_zip = False
else:
    url = SAMPLES_URL.strip()

    # GitHub sirve DOS urls distintas para el mismo fichero:
    #   .../blob/...  -> la pagina web que lo muestra  (HTML)
    #   .../raw/...   -> el fichero de verdad
    # Si te has copiado la de la barra del navegador, la arreglamos aqui.
    if "github.com" in url and "/blob/" in url:
        url = url.replace("/blob/", "/raw/")
        print("     ℹ  URL de GitHub corregida: /blob/ -> /raw/")
    url = url.replace("?raw=true", "").replace("?raw=1", "")

    if os.path.exists(DEST):
        os.remove(DEST)      # por si un intento anterior dejo un fichero malo

    if "drive.google.com" in url:
        _sh("pip install -q gdown")
        _sh("gdown --fuzzy '" + url + "' -O " + DEST)
    else:
        _sh("wget -q --no-check-certificate '" + url + "' -O " + DEST)

    # Comprobamos que lo descargado es DE VERDAD un ZIP mirando sus primeros
    # bytes. Que es, mira tu por donde, justo lo que vas a aprender hoy:
    # un ZIP siempre empieza por 50 4B 03 04, o sea "PK".
    cabecera = open(DEST, "rb").read(4) if os.path.exists(DEST) else b""
    ok_zip = cabecera == b"PK\x03\x04"

    if ok_zip:
        print("     ✔ Descargado (" + str(os.path.getsize(DEST)//1024) + " KB, empieza por 'PK' ✔)")
    else:
        print("     ✖ Lo que he descargado NO es un ZIP.")
        print("        Empieza por los bytes " + (cabecera.hex() or "(nada)") +
              " y un ZIP empieza siempre por 504b0304.")
        if b"<" in cabecera or b"\n" in cabecera:
            print("")
            print("        Parece una pagina HTML. Lo tipico: la URL apunta a la PAGINA")
            print("        de GitHub y no al fichero. Fijate en la diferencia:")
            print("           .../blob/main/...  <- pagina web   ✖")
            print("           .../raw/main/...   <- el fichero   ✔")
            print("        Pulsa el boton 'Raw' en GitHub y copia esa URL.")
        print("")
        print("        Alternativa: usa la celda de abajo, 'PLAN B'.")

print("3/4  Descomprimiendo (contraseña: infected)...")
import pyzipper
if ok_zip:
    try:
        with pyzipper.AESZipFile(DEST) as z:
            z.setpassword(PASSWORD.encode())
            z.extractall("/content/")
        print("     ✔ Descomprimido en /content/muestras/")
    except Exception as e:
        print("     ✖ Error al descomprimir:", e)

print("4/4  Comprobando el laboratorio...")
MUESTRAS = "/content/muestras"
ficheros = sorted(f for f in glob.glob(MUESTRAS + "/*") if not f.endswith("LEEME.txt"))
if len(ficheros) >= 9:
    print("     ✔ " + str(len(ficheros)) + " muestras listas")
    print("")
    print("=" * 52)
    print("   LABORATORIO LISTO  ✅")
    print("=" * 52)
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("     ✖ Solo encuentro " + str(len(ficheros)) + " ficheros. Usa la celda 'PLAN B' de abajo.")

print("""
⚠️  RECUERDA: esto son muestras REALES de malware.
    Estás dentro de una máquina virtual de Google que se destruye al cerrar.
    NO descargues estos ficheros a tu ordenador. NO los ejecutes.
    Hoy solo vamos a MIRARLOS, que es justo de lo que va el análisis estático.
""")

In [ ]:
#@title 📦 PLAN B — solo si la celda de arriba no ha conseguido las muestras { display-mode: "form" }
import glob, os

# Esta celda se puede ejecutar sola, sin haber pasado por la de arriba,
# asi que se instala ella misma lo que necesita.
try:
    import pyzipper
except ImportError:
    print("Instalando pyzipper (5 segundos)...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyzipper"], check=False)
    import pyzipper

ZIP_YA_SUBIDO = "/content/muestras_kschool.zip"

if os.path.exists(ZIP_YA_SUBIDO):
    # ya lo habias subido con el panel de Archivos de la izquierda
    print("Encontrado", ZIP_YA_SUBIDO, "- no hace falta que lo subas otra vez.")
    nombres = [ZIP_YA_SUBIDO]
else:
    from google.colab import files
    print("Pulsa en 'Elegir archivos' y selecciona muestras_kschool.zip")
    nombres = list(files.upload())

for nombre in nombres:
    try:
        with pyzipper.AESZipFile(nombre) as z:
            z.setpassword(b"infected")
            z.extractall("/content/")
    except Exception as e:
        print("✖ No he podido abrir", nombre, "->", e)

ficheros = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))
if ficheros:
    print("")
    print("✔ " + str(len(ficheros)) + " muestras listas en /content/muestras/")
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("✖ Sigo sin ver las muestras. Avisa en el chat.")

---
# Paso 1 · ¿Qué son las *strings*?

Un ejecutable es, sobre todo, código máquina: bytes que no significan nada para un humano.

Pero **entre medias** hay trozos de texto legible: mensajes de error, nombres de ficheros,
claves del registro, comandos, URLs… Todo lo que el programa necesita "escrito literalmente"
para hacer su trabajo.

A esos trozos los llamamos **strings**, y son, con diferencia, **la técnica más rentable del
análisis estático**: cinco minutos de leer strings te cuentan más de un malware que una hora
de mirar código.

La idea es ridículamente simple:

> *Recorre el fichero byte a byte. Cada vez que encuentres 4 o más caracteres imprimibles
> seguidos, eso es una string.*

Vamos a escribirla nosotros. Son dos líneas.

In [ ]:
import re

def strings_ascii(ruta, minimo=6):
    """Extrae textos ASCII de un fichero binario."""
    datos = open(ruta, "rb").read()
    patron = rb"[\x20-\x7e]{%d,}" % minimo      # \x20-\x7e = caracteres imprimibles
    return [m.decode() for m in re.findall(patron, datos)]

objetivo = "/content/muestras/curriculum_vitae_2024.pdf"
cadenas  = strings_ascii(objetivo)

print("He encontrado", len(cadenas), "cadenas de texto dentro del ejecutable.")
print()
print("Las 25 primeras:")
print("─" * 70)
for s in cadenas[:25]:
    print("  ", s)

Las primeras suelen ser aburridas (cabeceras del compilador). El clásico
`!This program cannot be run in DOS mode.` sale en **todos** los ejecutables de Windows:
es un mensajito de compatibilidad de 1993 que sigue ahí.

Lo interesante está más abajo. Y ahí aparece el primer problema real del análisis:
**hay demasiadas strings**. Más de 600. Nadie lee 600 líneas una por una.

> **La habilidad de verdad no es extraer strings. Es saber cuáles mirar.**

Vamos a filtrar.

---
# Paso 2 · Filtrar: buscar solo lo que delata

En vez de leerlas todas, buscamos **palabras que un programa normal no necesita**.

Piénsalo como un detective: una aplicación de contabilidad no tiene ninguna razón para
contener la palabra `netsh firewall`, ni `schtasks`, ni `AntiVirusProduct`.

In [ ]:
PISTAS = {
    "🔁 PERSISTENCIA (arrancar con Windows)": [
        "CurrentVersion\\Run", "RunOnce", "schtasks", "StartUp", "autorun.inf"],
    "🛡️ EVASIÓN (esquivar defensas)": [
        "netsh firewall", "AntiVirusProduct", "SecurityCenter", "Defender"],
    "🌐 RED": [
        "http", "ddns", "duckdns", "no-ip", "Socket", "Connect"],
    "🧹 ANTI-FORENSE (borrar el rastro)": [
        "del \"", "ping 0 -n", "DeleteFile", "TempFileName"],
    "🎭 OFUSCACIÓN": [
        "Base64String", "FromBase64", "ToBase64"],
    "⚙️ EJECUCIÓN DE COMANDOS": [
        "cmd.exe", "powershell", "rundll32", "WScript"],
}

for categoria, palabras in PISTAS.items():
    encontradas = sorted({s for s in cadenas
                          for p in palabras
                          if p.lower() in s.lower() and len(s) < 120})
    if encontradas:
        print(categoria)
        for s in encontradas[:8]:
            print("     ", repr(s))
        print()

### 🔎 Lo que ha salido… y sobre todo, lo que NO

Fíjate en **qué tipo** de cosas han aparecido:

| Lo que ves | Lo que significa |
|---|---|
| `System.Net.Sockets`, `GetActiveTcpConnections` | Sabe **abrir conexiones de red** |
| `FromBase64String`, `ToBase64String` | Sabe **codificar y decodificar Base64** |
| `GetTempFileName` | Sabe **crear ficheros temporales** |

Todo eso son **nombres de funciones del propio .NET** que el programa usa. Te dicen de qué
es **capaz**… pero no te dicen **qué hace**: ni a qué servidor se conecta, ni qué comandos
lanza, ni dónde se esconde.

Y ahora lo importante. Mira las categorías que han salido **vacías**:

> **Ni PERSISTENCIA. Ni EVASIÓN. Ni EJECUCIÓN DE COMANDOS.**

Un troyano que no se molesta en sobrevivir a un reinicio, que no toca el firewall y que no
ejecuta ni un solo comando. **Eso no se lo cree nadie.**

Quédate con esa pregunta. La respondemos en dos pasos.


---
# Paso 3 · Pero… ¿dónde está el C2?

Tenemos las técnicas. Nos falta **la dirección del servidor**. Búscala.

In [ ]:
import re

print("Buscando dominios y URLs entre las cadenas ASCII...")
print()

encontrado = False
for s in cadenas:
    if re.search(r"https?://|\.(com|net|org|ru|xyz|top|info|duckdns|ddns)\b", s, re.I):
        print("  ", repr(s))
        encontrado = True

if not encontrado:
    print("   (nada)")

print()
print("🤔 No hay ni un solo dominio. ¿Entonces cómo se conecta a su C2?")

### El detalle que se le escapa a mucha gente: **ASCII vs Unicode**

Nuestra función `strings_ascii` busca caracteres "normales": un byte = una letra.

Pero Windows, por dentro, guarda muchísimo texto en **UTF-16**, donde cada letra ocupa
**dos bytes**: la letra y un `00`.

Así que la palabra `hola` en memoria puede estar guardada así:

```
ASCII    68 6F 6C 61              →  "hola"     ✅ nuestra función la encuentra
UTF-16   68 00 6F 00 6C 00 61 00  →  "h·o·l·a"  ❌ nuestra función NO la ve
```

Para nuestro buscador, esos `00` rompen la cadena. **El texto está ahí, pero somos ciegos a él.**

> Esto en las herramientas reales se llama *wide strings*. En `strings.exe` de Sysinternals es
> la opción `-u`; en el `strings` de Linux es `-el`.
> **Si solo miras ASCII, te pierdes medio fichero.** Literalmente es el error número uno
> de quien empieza.

Vamos a añadir la búsqueda en UTF-16.

In [ ]:
import re

def strings_wide(ruta, minimo=6):
    """Extrae textos UTF-16 (wide) de un fichero binario."""
    datos = open(ruta, "rb").read()
    patron = rb"(?:[\x20-\x7e]\x00){%d,}" % minimo    # letra + 00, repetido
    return [m.decode("utf-16-le") for m in re.findall(patron, datos)]

anchas = strings_wide(objetivo)
print("Cadenas ASCII :", len(cadenas))
print("Cadenas WIDE  :", len(anchas), "  ← ¡estas no las estábamos viendo!")
print()

print("Cadenas wide que parecen sospechosas:")
print("─" * 70)
for s in anchas:
    if re.search(r"[A-Za-z0-9+/*!]{16,}", s) or re.search(r"\.(net|com|org)\b", s, re.I):
        print("  ", repr(s))

---
# Paso 4 · Ahora repite el filtro, con los ojos abiertos

Ya tenemos las cadenas wide. Vamos a pasar **el mismo filtro de antes**,
pero esta vez sobre las dos listas juntas.

In [ ]:
# EXACTAMENTE el mismo filtro del Paso 2. Lo único que cambia es dónde busca.
todas = cadenas + anchas

print("Antes buscábamos en", len(cadenas), "cadenas.  Ahora en", len(todas), ".")
print("─" * 70)
print()

for categoria, palabras in PISTAS.items():
    encontradas = sorted({s for s in todas
                          for p in palabras
                          if p.lower() in s.lower() and len(s) < 120})
    if encontradas:
        print(categoria)
        for s in encontradas[:8]:
            print("     ", repr(s))
        print()


### 🔎 Ahora sí. Léelo despacio, que aquí hay una historia entera

**No hemos cambiado el filtro: hemos cambiado dónde mira.** Las tres categorías que estaban
vacías se acaban de llenar.

Sin haber ejecutado nada y sin ser ingeniero inverso, ya sabemos **lo que hace este programa**:

| Lo que ves | Lo que significa |
|---|---|
| `Software\Microsoft\Windows\CurrentVersion\Run` | Se añade al arranque de Windows para sobrevivir a los reinicios |
| `schtasks /create /sc minute /mo 1 /tn StUpdate /tr` | Y por si acaso, **también** crea una tarea programada que se lanza **cada minuto** |
| `netsh firewall add allowedprogram` | Se abre un hueco en el firewall de Windows a sí mismo |
| `Select * From AntiVirusProduct` | Le pregunta a Windows **qué antivirus tienes instalado** |
| `cmd.exe /c ping 0 -n 2 & del "` | Truco clásico de autoborrado: espera dos pings y se borra solo |
| `autorun.inf` | Intenta propagarse a los USB que conectes |
| `\FransescoPast.txt` | Un fichero donde va guardando lo que captura (keylogger) |

> 😳 **Todo esto llevaba ahí desde el principio.** Estaba dentro del fichero mientras mirábamos
> la lista vacía y pensábamos que el malware "no hacía nada". Por eso el error de mirar solo
> ASCII es el número uno de quien empieza: no te da un error, te da una respuesta incompleta
> que parece completa.

Y ahora fíjate en algo distinto. Vuelve a mirar la lista de cadenas wide de un poco más arriba:
`EnviarDadosConexaooo`, `EnviarPermitirFormJanelas`, `ChamaFormProgramas`…

**Eso no es español. Es portugués.** *Enviar dados*, *chamar form*, *janelas* (ventanas).

Ese detalle no bloquea ningún ataque, pero es **oro puro para inteligencia de amenazas**:
te habla del idioma del desarrollador, y te permite **agrupar muestras del mismo autor**
aunque cambien el hash, el nombre y el servidor.

> Esto es exactamente el puente entre el análisis técnico y la criminología:
> el código es una escena del crimen, y el autor deja huellas culturales sin darse cuenta.


---
## 🧩 TU TURNO (5 minutos) — La cadena rara

Entre las cadenas wide hay una que canta muchísimo:

```
aGFraW0z*i5kZG5zLm5ldA!!
```

Eso tiene toda la pinta de estar en **Base64**, que es la forma más común de "disimular"
un texto. (Base64 **no es cifrado**: es solo otra forma de escribir lo mismo. Se usa para
que un texto no salte a la vista — ni en las strings, ni en un antivirus sencillo.)

Pero si intentas decodificarla directamente, **falla**. Porque tiene dos caracteres que el
Base64 normal no usa nunca: `*` y `!`.

> 💡 **La pista:** esta familia de malware hace una sustitución muy tonta para despistar.
> Cambia el carácter `=` por `!`, y el carácter `M` por `*`.
> Deshaz esa sustitución **antes** de decodificar.

**No tienes que escribir código.** En la celda de abajo solo hay que rellenar cuatro
casillas: qué dos caracteres sobran, y por cuáles hay que cambiarlos. Luego pulsa ▶️.


In [ ]:
#@title 🧩 TU TURNO — deshaz la sustitución y pulsa ▶️ { display-mode: "form" }

#@markdown Rellena las cuatro casillas con **un solo carácter** cada una.
#@markdown Si te equivocas no pasa nada: la celda te dice qué falta y puedes volver a probar.

caracter_que_sobra_1 = "" #@param {type:"string"}
hay_que_cambiarlo_por_1 = "" #@param {type:"string"}
caracter_que_sobra_2 = "" #@param {type:"string"}
hay_que_cambiarlo_por_2 = "" #@param {type:"string"}

# ───────── a partir de aquí no hay que tocar nada ─────────
import base64

sospechosa = "aGFraW0z*i5kZG5zLm5ldA!!"

limpia = sospechosa
if caracter_que_sobra_1:
    limpia = limpia.replace(caracter_que_sobra_1, hay_que_cambiarlo_por_1)
if caracter_que_sobra_2:
    limpia = limpia.replace(caracter_que_sobra_2, hay_que_cambiarlo_por_2)

print("Original :", sospechosa)
print("Limpia   :", limpia)
print()

if not caracter_que_sobra_1 and not caracter_que_sobra_2:
    print("✋ Todavía no has rellenado nada.")
    print("   Mira la cadena de arriba: ¿qué dos caracteres NO pegan en un Base64?")
else:
    try:
        resultado = base64.b64decode(limpia).decode()
        print("🎯 DECODIFICADO:", resultado)
        print()
        print("   Eso es el C2: el servidor al que este malware llama a casa.")
    except Exception:
        print("✖ Todavía no es un Base64 válido.")
        sobran = sorted({c for c in limpia if not (c.isalnum() or c in "+/=")})
        if sobran:
            print("   Siguen sobrando estos caracteres:", "  ".join(sobran))
        else:
            print("   Ya no sobra ningún carácter, pero el relleno del final no cuadra.")
        print("   Recuerda la pista:  el atacante cambió  '='  por  '!'   y   'M'  por  '*'")


---
## 🎉 Lo tienes

Acabas de hacer, en diez minutos y sin ejecutar nada, lo que en un incidente real se reporta así:

```
Muestra    : curriculum_vitae_2024.pdf (en realidad PE32 .NET)
SHA-256    : 3b7c80a670ed7981e02530ca4fc4ff52e46ebe19e6ddaf3fde249d25918da77b
Familia    : njRAT / Bladabindi
C2         : hakim32.ddns.net                         ← BLOQUEAR EN FIREWALL Y DNS
Persistencia: HKCU\...\CurrentVersion\Run  +  tarea "StUpdate" (cada minuto)
Evasión    : netsh firewall add allowedprogram · consulta WMI AntiVirusProduct
Propagación: autorun.inf en unidades extraíbles
Atribución : cadenas de código en PORTUGUÉS
```

Ese bloque es, literalmente, lo que un analista de CTI mete en un informe o en un MISP.
Y sale entero del análisis estático.

> 🔗 **Pivotar:** con ese dominio puedes ir a VirusTotal y buscar
> `hakim32.ddns.net` para ver **qué otras muestras** se conectan al mismo sitio.
> Un IOC nunca está solo: tirando del hilo aparece toda la campaña.

---
# 🏆 RETO (si vas sobrado)

## Reto A · Tu extractor de IOCs automático

En un SOC nadie hace esto a mano. Escribe algo que, dado un fichero, escupa directamente
la lista de indicadores.

Completa el diccionario de expresiones regulares y ejecútalo sobre **todas** las muestras.

In [ ]:
import glob, os, re

IOC_REGEX = {
    "URL":      r"https?://[^\s\"\'<>]{6,}",
    "IPv4":     r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
    "DOMINIO":  r"\b[a-z0-9-]{3,}\.(?:com|net|org|ru|info|xyz|top|online|duckdns\.org|ddns\.net|no-ip\.org|hopto\.org)\b",
    "REGISTRO": r"(?:HKEY_[A-Z_]+|Software)\\[^\s\"\'<>]{4,70}",
    "RUTA":     r"[A-Za-z]:\\[^\s\"\'<>|*?]{3,60}",
    "EMAIL":    r"[\w.+-]+@[\w-]+\.[\w.]{2,}",
    # 👇 AÑADE TÚ UNO MÁS: por ejemplo, para detectar carteras de criptomonedas,
    #    mutex, o nombres de tareas programadas
}

def extraer_iocs(ruta):
    todas = set(strings_ascii(ruta, 5)) | set(strings_wide(ruta, 5))
    hallazgos = {}
    for nombre, patron in IOC_REGEX.items():
        h = set()
        for s in todas:
            h.update(re.findall(patron, s, re.I))
        if h:
            hallazgos[nombre] = sorted(h)
    return hallazgos

for f in sorted(glob.glob("/content/muestras/*.exe")) + [objetivo]:
    print("=" * 78)
    print("📄", os.path.basename(f))
    iocs = extraer_iocs(f)
    if not iocs:
        print("   (sin indicadores en texto plano — probablemente esté empaquetado)")
    for tipo, valores in iocs.items():
        print("   {:<10} {}".format(tipo, ", ".join(valores[:4])[:150]))

### Reto B · ¿Por qué `actualizacion_flash.exe` no suelta nada?

Ejecuta el extractor sobre `actualizacion_flash.exe` (el empaquetado) y sobre su gemelo
`actualizacion_flash_DESEMPAQUETADO.bin`. Cuenta las strings de cada uno.

In [10]:
for nombre in ["actualizacion_flash.exe", "actualizacion_flash_DESEMPAQUETADO.bin"]:
    ruta = "/content/muestras/" + nombre
    n_ascii = len(strings_ascii(ruta))
    n_wide  = len(strings_wide(ruta))
    print("{:<42} ascii={:>6}   wide={:>6}".format(nombre, n_ascii, n_wide))

print()
print("👉 Mira la columna WIDE: 53 frente a más de 1.200.")
print("   No es que el programa empaquetado no tenga esas cadenas: es que están")
print("   comprimidas y cifradas dentro de la sección UPX1, y solo aparecen cuando")
print("   el programa se ejecuta y se descomprime a sí mismo en memoria.")
print()
print("   (La columna ASCII apenas cambia porque UPX deja la sección de recursos")
print("    .rsrc sin comprimir: iconos, textos de la interfaz... ruido, básicamente.")
print("    Otra razón más para mirar SIEMPRE las dos columnas.)")
print()
print("   ESTA es la razón número uno por la que el malware usa packers:")
print("   no para ser más pequeño, sino para que el análisis estático no vea nada.")
print()
print("   ¿Y qué hace entonces el analista? Dos opciones:")
print("     1. Desempaquetarlo (estático avanzado)")
print("     2. Ejecutarlo en un sandbox y volcar la memoria (análisis dinámico)")
print("   → y de eso va precisamente la sesión siguiente del módulo.")

actualizacion_flash.exe                    ascii=  1632   wide=    53
actualizacion_flash_DESEMPAQUETADO.bin     ascii=  1884   wide=  1217

👉 Mira la columna WIDE: 53 frente a más de 1.200.
   No es que el programa empaquetado no tenga esas cadenas: es que están
   comprimidas y cifradas dentro de la sección UPX1, y solo aparecen cuando
   el programa se ejecuta y se descomprime a sí mismo en memoria.

   (La columna ASCII apenas cambia porque UPX deja la sección de recursos
    .rsrc sin comprimir: iconos, textos de la interfaz... ruido, básicamente.
    Otra razón más para mirar SIEMPRE las dos columnas.)

   ESTA es la razón número uno por la que el malware usa packers:
   no para ser más pequeño, sino para que el análisis estático no vea nada.

   ¿Y qué hace entonces el analista? Dos opciones:
     1. Desempaquetarlo (estático avanzado)
     2. Ejecutarlo en un sandbox y volcar la memoria (análisis dinámico)
   → y de eso va precisamente la sesión siguiente del módulo.


### Reto C · FLOSS, o cómo hacer esto de verdad en un SOC

Hay una herramienta de Mandiant/Google llamada **FLOSS** (*FLARE Obfuscated String Solver*)
que hace lo que hemos hecho nosotros **y además** deshace automáticamente muchas ofuscaciones:
emula pequeños trozos de código para recuperar strings que el malware construye en tiempo de
ejecución.

La lógica es la misma que la que acabas de escribir. Solo que con diez años de trabajo encima.

- 🔗 https://github.com/mandiant/flare-floss

> **Lo importante no es la herramienta.** Es que ahora entiendes *qué* hace por dentro
> y *por qué* a veces no encuentra nada (spoiler: porque estaba empaquetado).

---
# ✅ Antes de la puesta en común

1. 🌐 El C2 es: **`________________________`**
2. 🔧 Dos técnicas del atacante: **`________________`** y **`________________`**
3. 🌍 Bonus: ¿en qué idioma programaba el autor? **`__________`**

**Siguiente:** LAB 3 · *El adjunto sospechoso* — vamos a por la factura en Word y los dos PDFs.